In [0]:
# Load bronze layer tables - Synthetic tables
bronze_sales = spark.table("supply_chain.bronze.sales")
bronze_po = spark.table("supply_chain.bronze.purchase_orders")
bronze_inventory = spark.table("supply_chain.bronze.inventory")
bronze_loads = spark.table("supply_chain.bronze.loads")

# Load Kaggle CSV tables from Bronze
bronze_customers = spark.table("supply_chain.bronze.customers")
bronze_drivers = spark.table("supply_chain.bronze.drivers")
bronze_driver_monthly_metrics = spark.table("supply_chain.bronze.driver_monthly_metrics")
bronze_delivery_events = spark.table("supply_chain.bronze.delivery_events")
bronze_facilities = spark.table("supply_chain.bronze.facilities")
bronze_fuel_purchases = spark.table("supply_chain.bronze.fuel_purchases")
bronze_maintenance_records = spark.table("supply_chain.bronze.maintenance_records")
bronze_routes = spark.table("supply_chain.bronze.routes")
bronze_safety_incidents = spark.table("supply_chain.bronze.safety_incidents")
bronze_trailers = spark.table("supply_chain.bronze.trailers")
bronze_trips = spark.table("supply_chain.bronze.trips")
bronze_truck_utilization_metrics = spark.table("supply_chain.bronze.truck_utilization_metrics")
bronze_trucks = spark.table("supply_chain.bronze.trucks")

print(f"Bronze Sales: {bronze_sales.count()} rows")
print(f"Bronze Purchase Orders: {bronze_po.count()} rows")
print(f"Bronze Inventory: {bronze_inventory.count()} rows")
print(f"Bronze Loads: {bronze_loads.count()} rows")
print(f"Bronze Customers: {bronze_customers.count()} rows")
print(f"Bronze Drivers: {bronze_drivers.count()} rows")
print(f"Bronze Driver Monthly Metrics: {bronze_driver_monthly_metrics.count()} rows")
print(f"Bronze Delivery Events: {bronze_delivery_events.count()} rows")
print(f"Bronze Facilities: {bronze_facilities.count()} rows")
print(f"Bronze Fuel Purchases: {bronze_fuel_purchases.count()} rows")
print(f"Bronze Maintenance Records: {bronze_maintenance_records.count()} rows")
print(f"Bronze Routes: {bronze_routes.count()} rows")
print(f"Bronze Safety Incidents: {bronze_safety_incidents.count()} rows")
print(f"Bronze Trailers: {bronze_trailers.count()} rows")
print(f"Bronze Trips: {bronze_trips.count()} rows")
print(f"Bronze Truck Utilization Metrics: {bronze_truck_utilization_metrics.count()} rows")
print(f"Bronze Trucks: {bronze_trucks.count()} rows")

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date, when, current_timestamp

bronze_po = spark.table("workspace.default.purchase_orders")

silver_po = (
    bronze_po

    # 1. Trim spaces
    .withColumn("p_id", trim(col("p_id")))
    .withColumn("sup_id", trim(col("sup_id")))

    # 2. Fix date format
    .withColumn("order_date", to_date(col("order_date")))

    # 3. Standardization
    .withColumn("p_id", upper(col("p_id")))
    .withColumn("sup_id", upper(col("sup_id")))

    # 4. Standardize status (if exists)
    .withColumn(
        "order_status",
        when(col("received_qty") == col("ordered_qty"), "COMPLETED")
        .when(col("received_qty") == 0, "PENDING")
        .otherwise("PARTIAL")
    )

    # 5. Remove duplicates
    .dropDuplicates(["p_id"])

    # 6. Handle null values
    .fillna({
        "received_qty": 0
    })
    .dropna(subset=["p_id", "sup_id", "order_date"])

    # Validation
    .filter((col("ordered_qty") >= 0) & (col("received_qty") >= 0))

    # Derived column
    .withColumn("open_qty", col("ordered_qty") - col("received_qty"))

    # Metadata
    .withColumn("processed_ts", current_timestamp())
)

# 8. Save as Delta
silver_po.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.silver_purchase_orders")

### Sales

In [0]:
bronze_sales = spark.table("workspace.default.sales")

silver_sales = (
    bronze_sales

    # Trim
    .withColumn("sku_id", trim(col("sku_id")))
    .withColumn("location_id", trim(col("location_id")))

    # Fix date
    .withColumn("order_date", to_date(col("order_date")))

    # Standardization
    .withColumn("sku_id", upper(col("sku_id")))
    .withColumn("location_id", upper(col("location_id")))

    # Remove duplicates
    .dropDuplicates(["sales_id"])

    # Handle nulls
    .fillna({
        "unit_price": 0
    })

    .dropna(subset=["sales_id", "order_id", "customer_id"])

    # Validation
    .filter(col("quantity_sold") > 0)

    # Fix total_amount
    .withColumn("total_amount", col("quantity_sold") * col("unit_price"))

    .withColumn("processed_ts", current_timestamp())
)

silver_sales.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.silver_sales")

In [0]:
bronze_loads = spark.table("workspace.default.loads")

silver_loads = (
    bronze_loads

    # Remove duplicates
    .dropDuplicates()

    # Handle nulls
    .dropna(how="all")

    # Metadata
    .withColumn("processed_ts", current_timestamp())
)

silver_loads.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.silver_loads")

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date, to_timestamp, coalesce, when, current_timestamp
from pyspark.sql.types import StringType

# -------------------------
# 1. Trim all string columns
# -------------------------
def trim_all_string_columns(df):
    schema = df.schema
    columns = df.columns
    
    trim_columns = {
        column: trim(col(column)) 
        for column in columns 
        if isinstance(schema[column].dataType, StringType)
    }
    
    if trim_columns:
        df = df.withColumns(trim_columns)
    return df

# -------------------------
# 2. Parse timestamps with multiple formats
# -------------------------
def parse_timestamp(column):
    return coalesce(
        to_timestamp(col(column), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col(column), "yyyy-MM-dd'T'HH:mm:ss"),
        to_timestamp(col(column), "dd-MM-yyyy HH:mm:ss"),
        to_timestamp(col(column), "yyyy/MM/dd HH:mm:ss"),
        to_timestamp(col(column), "dd/MM/yyyy HH:mm:ss")
    )

# -------------------------
# 3. Standardize strings (uppercase + trim)
# -------------------------
def standardize_strings(df):
    return df.select([
        upper(trim(col(c))).alias(c) if isinstance(df.schema[c].dataType, StringType)
        else col(c)
        for c in df.columns
    ])

print("✅ Utility functions defined successfully!")

In [0]:
# ============================================
# SILVER TRANSFORMATIONS FOR ALL 17 TABLES
# ============================================

# -----------------
# 1. SALES
# -----------------
silver_sales = (
    bronze_sales
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["sales_id"])
    .fillna({"unit_price": 0.0})
    .dropna(subset=["sales_id", "order_id", "customer_id"])
    .filter(col("quantity_sold") > 0)
    .withColumn("total_amount", col("quantity_sold") * col("unit_price"))
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 2. PURCHASE ORDERS
# -----------------
silver_purchase_orders = (
    bronze_po
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .withColumn(
        "order_status",
        when(col("received_qty") == col("ordered_qty"), "COMPLETED")
        .when(col("received_qty") == 0, "PENDING")
        .otherwise("PARTIAL")
    )
    .dropDuplicates(["p_id"])
    .fillna({"received_qty": 0})
    .dropna(subset=["p_id", "sup_id", "order_date"])
    .filter((col("ordered_qty") >= 0) & (col("received_qty") >= 0))
    .withColumn("open_qty", col("ordered_qty") - col("received_qty"))
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 3. INVENTORY
# -----------------
silver_inventory = (
    bronze_inventory
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["sku_id", "location_id"])
    .fillna({"on_hand_quantity": 0, "intransit_qty": 0})
    .filter((col("on_hand_quantity") >= 0) & (col("intransit_qty") >= 0))
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 4. LOADS
# -----------------
silver_loads = (
    bronze_loads
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["load_id"])
    .fillna({"weight_lbs": 0, "pieces": 0, "revenue": 0.0, "fuel_surcharge": 0.0, "accessorial_charges": 0})
    .withColumn(
        "load_status",
        when(col("load_status").isin("DELIVERED", "IN_TRANSIT", "CANCELED", "SCHEDULED"), col("load_status"))
        .otherwise("OTHER")
    )
    .dropna(subset=["load_id", "customer_id"])
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 5. CUSTOMERS
# -----------------
silver_customers = (
    bronze_customers
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["customer_id"])
    .fillna({"customer_name": "UNKNOWN", "customer_type": "UNKNOWN", "account_status": "UNKNOWN", "credit_terms_days": 0, "annual_revenue_potential": 0})
    .dropna(subset=["customer_id"])
    .withColumn(
        "account_status",
        when(col("account_status").isin("ACTIVE", "INACTIVE", "SUSPENDED"), col("account_status"))
        .otherwise("UNKNOWN")
    )
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 6. DRIVERS
# -----------------
silver_drivers = (
    bronze_drivers
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["driver_id"])
    .fillna({"first_name": "UNKNOWN", "last_name": "UNKNOWN", "license_state": "UNKNOWN", "years_experience": 0})
    .dropna(subset=["driver_id", "license_number"])
    .withColumn(
        "employment_status",
        when(col("employment_status").isin("ACTIVE", "TERMINATED", "ON_LEAVE"), col("employment_status"))
        .otherwise("UNKNOWN")
    )
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 7. DRIVER MONTHLY METRICS
# -----------------
silver_driver_monthly_metrics = (
    bronze_driver_monthly_metrics
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["driver_id", "month"])
    .fillna({"trips_completed": 0, "total_miles": 0, "total_revenue": 0.0, "average_mpg": 0.0, "total_fuel_gallons": 0.0})
    .dropna(subset=["driver_id", "month"])
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 8. DELIVERY EVENTS
# -----------------
silver_delivery_events = (
    bronze_delivery_events
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["event_id"])
    .fillna({"event_type": "UNKNOWN", "location_city": "UNKNOWN", "location_state": "UNKNOWN"})
    .dropna(subset=["event_id", "load_id"])
    .withColumn(
        "event_type",
        when(col("event_type").isin("PICKUP", "DELIVERY", "IN_TRANSIT", "DELAY"), col("event_type"))
        .otherwise("OTHER")
    )
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 9. FACILITIES
# -----------------
silver_facilities = (
    bronze_facilities
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["facility_id"])
    .fillna({"facility_name": "UNKNOWN", "facility_type": "UNKNOWN", "city": "UNKNOWN", "state": "UNKNOWN"})
    .dropna(subset=["facility_id"])
    .withColumn(
        "facility_type",
        when(col("facility_type").isin("WAREHOUSE", "DISTRIBUTION_CENTER", "TERMINAL"), col("facility_type"))
        .otherwise("OTHER")
    )
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 10. FUEL PURCHASES (Fixed: fuel_purchase_id not purchase_id)
# -----------------
silver_fuel_purchases = (
    bronze_fuel_purchases
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["fuel_purchase_id"])
    .fillna({"gallons": 0.0, "price_per_gallon": 0.0, "total_cost": 0.0, "fuel_card_number": "UNKNOWN"})
    .dropna(subset=["fuel_purchase_id", "truck_id"])
    .filter((col("gallons") >= 0) & (col("price_per_gallon") >= 0))
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 11. MAINTENANCE RECORDS (Fixed: total_cost, service_description)
# -----------------
silver_maintenance_records = (
    bronze_maintenance_records
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["maintenance_id"])
    .fillna({"maintenance_type": "UNKNOWN", "total_cost": 0.0, "service_description": "UNKNOWN", "labor_cost": 0.0, "parts_cost": 0.0})
    .dropna(subset=["maintenance_id", "truck_id"])
    .filter(col("total_cost") >= 0)
    .withColumn(
        "maintenance_type",
        when(col("maintenance_type").isin("SCHEDULED", "UNSCHEDULED", "EMERGENCY"), col("maintenance_type"))
        .otherwise("OTHER")
    )
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 12. ROUTES (Fixed: typical_distance_miles, typical_transit_days)
# -----------------
silver_routes = (
    bronze_routes
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["route_id"])
    .fillna({"origin_city": "UNKNOWN", "destination_city": "UNKNOWN", "typical_distance_miles": 0, "typical_transit_days": 0})
    .dropna(subset=["route_id"])
    .filter((col("typical_distance_miles") >= 0) & (col("typical_transit_days") >= 0))
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 13. SAFETY INCIDENTS
# -----------------
silver_safety_incidents = (
    bronze_safety_incidents
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["incident_id"])
    .fillna({"incident_type": "UNKNOWN", "description": "UNKNOWN", "at_fault_flag": False, "injury_flag": False, "preventable_flag": False})
    .dropna(subset=["incident_id"])
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 14. TRAILERS (Fixed: length_feet, model_year)
# -----------------
silver_trailers = (
    bronze_trailers
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["trailer_id"])
    .fillna({"trailer_type": "UNKNOWN", "length_feet": 0, "model_year": 0, "status": "UNKNOWN"})
    .dropna(subset=["trailer_id"])
    .withColumn(
        "status",
        when(col("status").isin("AVAILABLE", "IN_USE", "MAINTENANCE", "RETIRED"), col("status"))
        .otherwise("UNKNOWN")
    )
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 15. TRIPS (Fixed: actual_distance_miles, actual_duration_hours, fuel_gallons_used)
# -----------------
silver_trips = (
    bronze_trips
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["trip_id"])
    .fillna({"actual_distance_miles": 0, "actual_duration_hours": 0.0, "fuel_gallons_used": 0.0, "average_mpg": 0.0, "idle_time_hours": 0.0})
    .dropna(subset=["trip_id", "truck_id", "driver_id"])
    .filter((col("actual_distance_miles") >= 0) & (col("actual_duration_hours") >= 0) & (col("fuel_gallons_used") >= 0))
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 16. TRUCK UTILIZATION METRICS
# -----------------
silver_truck_utilization_metrics = (
    bronze_truck_utilization_metrics
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["truck_id", "month"])
    .fillna({"trips_completed": 0, "total_miles": 0, "total_revenue": 0.0, "maintenance_cost": 0.0})
    .dropna(subset=["truck_id", "month"])
    .filter((col("total_miles") >= 0) & (col("total_revenue") >= 0))
    .withColumn("processed_ts", current_timestamp())
)

# -----------------
# 17. TRUCKS (Fixed: model_year)
# -----------------
silver_trucks = (
    bronze_trucks
    .transform(trim_all_string_columns)
    .transform(standardize_strings)
    .dropDuplicates(["truck_id"])
    .fillna({"make": "UNKNOWN", "model_year": 0, "status": "UNKNOWN", "fuel_type": "UNKNOWN"})
    .dropna(subset=["truck_id"])
    .withColumn(
        "status",
        when(col("status").isin("ACTIVE", "MAINTENANCE", "RETIRED", "INACTIVE"), col("status"))
        .otherwise("UNKNOWN")
    )
    .withColumn("processed_ts", current_timestamp())
)

print("✅ All 17 silver transformations completed!")
print(f"\nSilver Sales: {silver_sales.count()} rows")
print(f"Silver Purchase Orders: {silver_purchase_orders.count()} rows")
print(f"Silver Inventory: {silver_inventory.count()} rows")
print(f"Silver Loads: {silver_loads.count()} rows")

In [0]:
# ============================================
# WRITE ALL 17 SILVER TABLES TO CATALOG
# ============================================

silver_tables = {
    "silver_sales": silver_sales,
    "silver_purchase_orders": silver_purchase_orders,
    "silver_inventory": silver_inventory,
    "silver_loads": silver_loads,
    "silver_customers": silver_customers,
    "silver_drivers": silver_drivers,
    "silver_driver_monthly_metrics": silver_driver_monthly_metrics,
    "silver_delivery_events": silver_delivery_events,
    "silver_facilities": silver_facilities,
    "silver_fuel_purchases": silver_fuel_purchases,
    "silver_maintenance_records": silver_maintenance_records,
    "silver_routes": silver_routes,
    "silver_safety_incidents": silver_safety_incidents,
    "silver_trailers": silver_trailers,
    "silver_trips": silver_trips,
    "silver_truck_utilization_metrics": silver_truck_utilization_metrics,
    "silver_trucks": silver_trucks
}

for table_name, df in silver_tables.items():
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"supply_chain.silver.{table_name}")
    print(f"✓ Written {table_name} to supply_chain.silver.{table_name}")

print(f"\n✅ All {len(silver_tables)} silver tables written successfully!")